[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.5 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn

In [5]:
'''
如果 batch 很大，显存不够，就没法一次把整个 batch 都塞进去
把一个大 batch 拆成多个 micro_batches
每次只跑一个小 batch
每次都 backward()
梯度会自动累加到参数的 .grad 上
最后统一 optimizer.step()
'''

def accumulated_step(model, optimizer, loss_fn, micro_batches):
  # PyTorch 默认梯度是累加的，不会自动清零。 每次新的训练 step 开始前，要先清掉上一次留下来的梯度。
    optimizer.zero_grad()

    total_loss = 0.0
    n = len(micro_batches)

    for x, y in micro_batches:
      pred = model(x)
      loss = loss_fn(pred, y)
      total_loss += loss.item() # .item() 是把 tensor 变成 Python float

      (loss / n).backward() # 反向传播 会进行n次
      # 这不会覆盖.grad 而是累加起来

    # 前面每个 micro-batch 的梯度都已经累加进 .grad 里了， 现在同意更新一次参数
    optimizer.step()

    return total_loss / n

In [6]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 1.2422220930457115


In [7]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (36.4ms)
  ✅ [2/3] Returns loss value (2.0ms)
  ✅ [3/3] Parameters actually update (1.6ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (40.0ms total)
  Progress saved. Run status() to see your dashboard.

